## NLP Feature Extraction Pipeline
This notebook loads the raw, uncleaned scraped data and processes the text fields to generate lightweight NLP features for the Streamlit dashboard.

In [1]:
!pip install textstat
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 72.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [textstat]


In [3]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from transformers import pipeline
import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

data_path = "../data/cars_and_bids_full_history_v3.csv"
df = pd.read_csv(data_path)

text_cols = [col for col in ['URL', 'Make', 'Model', 'Sold_Price', 'Modifications', 'Equipment', 'Highlights', 'Known Flaws', 'Recent_Service History', 'Ownership History', 'Other Items Included in Sale', 'Seller Notes'] if col in df.columns]
display(df[text_cols].head(5))

2026-03-13 03:08:54.096963: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


,URL,Make,Model,Sold_Price,Modifications,Equipment,Highlights,Known Flaws,Ownership History,Other Items Included in Sale,Seller Notes
0,https://carsandbids.com/auctions/rkdqRPJN/2026...,Mercedes-Benz,G Wagen\nSave,226000.0,NaN,A build sheet is provided in the photo gallery...,"THIS… is a 2026 Mercedes-AMG G63, finished in ...",NaN,The seller is representing this G63 on behalf ...,2 keys Owner's manuals,"There is a loan on this vehicle, and sale proc..."
1,https://carsandbids.com/auctions/30Op4L4g/2026...,BMW,G8X M4\nSave,76500.0,NaN,"A build sheet is provided in the gallery, and ...","THIS... is a 2026 BMW M4 Coupe, finished in Fr...",NaN,The seller reportedly purchased this M4 new in...,2 keys Owner's manual Window sticker,The seller states that clear paint protection ...
2,https://carsandbids.com/auctions/3010V6BV/2026...,Mercedes-Benz,G Wagen\nSave,215485.0,NaN,A build sheet is provided in the photo gallery...,"THIS… is a 2026 Mercedes-AMG G63, finished in ...",NaN,The seller reportedly purchased this G63 new i...,2 keys Owner's manual Build sheet All-weather ...,"There is a loan on this vehicle, and sale proc..."
3,https://carsandbids.com/auctions/KZB8oVXk/2026...,Porsche,992 911\nSave,356000.0,NaN,A build sheet is provided in the photo gallery...,"THIS... is a 2026 Porsche 911 GT3 Touring, fin...",NaN,The selling dealer reportedly acquired this 91...,2 keys Owner's manual Build sheet,NaN
4,https://carsandbids.com/auctions/rxVbB5lO/bruc...,THIS... is a charity auction for a Bruce Canep...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1. Entity Recognition: Premium Aftermarket Brands
Extracts specific high-value brands from the `Modifications` column.


In [4]:
brands = [
    "bilstein", "ohlins", "bbs", "recaro", "dinan", 
    "kw", "michelin", "koni", "brembo", "borla", "bosch",
    "apr", "momo", "nardi", "holley", "edgemotorsport", "moog", "acdelco", "hks"
]

def extract_brands(text):
    if pd.isna(text) or not isinstance(text, str):
        return []
    
    text_lower = text.lower()
    found_brands = []
    
    for brand in brands:
        if re.search(rf'\b{brand}\b', text_lower):
            found_brands.append(brand.capitalize())
            
    return found_brands

if 'Modifications' in df.columns:
    print("Extracting premium brands...")
    df['Extracted_Brands'] = df['Modifications'].apply(extract_brands)
    df['Has_Premium_Mods'] = df['Extracted_Brands'].apply(lambda x: 1 if len(x) > 0 else 0)
    df['Premium_Brand_Count'] = df['Extracted_Brands'].apply(len)
    df['Extracted_Brands_List'] = df['Extracted_Brands'].apply(lambda x: ", ".join(x))
    
    # Preview listings that actually have premium mods
    display(df[df['Has_Premium_Mods'] == 1][['Make', 'Model', 'Modifications', 'Extracted_Brands_List']].head())
else:
    print("Column 'Modifications' not found.")

Extracting premium brands...


,Make,Model,Modifications,Extracted_Brands_List
94,BMW,8 Series\nSave,Notable modifications reported by the seller i...,Apr
104,Toyota,Supra\nSave,Notable modifications reported by the seller i...,Hks
108,BMW,2 Series\nSave,Notable modifications reported by the seller i...,Bbs
129,Ford,Mustang\nSave,Notable modifications performed by Shelby Amer...,Borla
130,Ford,F-150\nSave,Notable modifications reported by the seller i...,Bilstein


### 2. Topic Modeling: Listing Archetypes
Uses Non-Negative Matrix Factorization (NMF) to cluster cars based on the combined text of their Highlights and Recent Service history.

In [ ]:
if 'Highlights' in df.columns and 'Recent_Service' in df.columns:
    print("Combining text and fitting NMF model...")
    # Combine text fields, filling NaNs
    text_data = df['Highlights'].fillna("") + " " + df['Recent_Service'].fillna("")
    text_list = text_data.tolist()
    
    # Vectorize
    tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words='english')
    tfidf_matrix = tfidf_vectorizer.fit_transform(text_list)
    
    # Fit NMF (adjust n_components to find the best archetypes)
    N_TOPICS = 4
    nmf_model = NMF(n_components=N_TOPICS, random_state=42, init='nndsvda')
    nmf_features = nmf_model.fit_transform(tfidf_matrix)
    
    df['Archetype_Cluster'] = nmf_features.argmax(axis=1)
    
    # Display the keywords for each cluster so you can manually interpret them
    feature_names = tfidf_vectorizer.get_feature_names_out()
    print("\n--- Cluster Keywords ---")
    for topic_idx, topic in enumerate(nmf_model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-10 - 1:-1]]
        print(f"Cluster {topic_idx}: {', '.join(top_words)}")
        
    # Preview
    display(df[['Make', 'Model', 'Highlights', 'Archetype_Cluster']].head())
else:
    print("Required text columns for Topic Modeling not found.")

## 3. Seller Persona: Zero-Shot Classification
Uses a Hugging Face transformer model to categorize the seller's tone.

In [ ]:
# Initialize the pipeline (downloads model weights on first run)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

PERSONA_LABELS = [
    "Passionate Enthusiast", 
    "Clinical Dealership", 
    "Reluctant Seller",
    "Flipping for Profit"
]

def get_seller_persona(text):
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) < 10:
        return "Unknown"
        
    # Truncate to avoid exceeding max token lengths
    truncated_text = text[:1000] 
    result = classifier(truncated_text, PERSONA_LABELS)
    return result['labels'][0]

if 'Highlights' in df.columns:
    # OPTIONAL: Test on a small sample first before running on the whole dataset
    sample_size = 5
    print(f"Testing zero-shot classification on a sample of {sample_size} rows...")
    
    sample_df = df.head(sample_size).copy()
    sample_df['Seller_Persona'] = sample_df['Highlights'].apply(get_seller_persona)
    display(sample_df[['Make', 'Model', 'Highlights', 'Seller_Persona']])
    
    # UNCOMMENT the lines below to run on the entire dataset
    # print("Running classification on the full dataset...")
    # df['Seller_Persona'] = df['Highlights'].apply(get_seller_persona)
else:
    print("Column 'Highlights' not found.")

## 4. Seller Transparency and "Effort" Scoring
Calculates word count, structural density (bullet points), readability, and sentiment across the main text fields to gauge seller effort.

In [2]:
analyzer = SentimentIntensityAnalyzer()

def calculate_effort_metrics(text):
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) == 0:
        return 0, 0, 0, 0
    
    # 1. Length/Detail Volume
    word_count = textstat.lexicon_count(text, removepunct=True)
    
    # 2. Readability (Flesch Reading Ease: higher is easier to read)
    readability = textstat.flesch_reading_ease(text)
    
    # 3. Structural Density (Bullet points imply organized data)
    bullet_points = text.count('- ') + text.count('* ') + text.count('•')
    
    # 4. Sentiment (Compound score from -1 to 1)
    sentiment_dict = analyzer.polarity_scores(text)
    compound_sentiment = sentiment_dict['compound']
    
    return word_count, readability, bullet_points, compound_sentiment

# Combine the main descriptive text fields to evaluate total effort
text_cols_to_check = ['Highlights', 'Known_Flaws', 'Recent_Service']
if all(col in df.columns for col in text_cols_to_check):
    print("Calculating effort metrics...")
    
    # Create a combined text column for analysis
    df['Combined_Seller_Text'] = df['Highlights'].fillna("") + "\n" + \
                                 df['Known_Flaws'].fillna("") + "\n" + \
                                 df['Recent_Service'].fillna("")
    
    # Apply the scoring function
    metrics = df['Combined_Seller_Text'].apply(calculate_effort_metrics)
    
    # Expand the returned tuples into distinct columns
    df[['Word_Count', 'Readability_Score', 'Bullet_Point_Count', 'Sentiment_Score']] = pd.DataFrame(metrics.tolist(), index=df.index)
    
    # Create a simple composite score (adjust weights as you see fit)
    # E.g., 1 point per word, 10 points per organized bullet point
    df['Composite_Effort_Score'] = df['Word_Count'] + (df['Bullet_Point_Count'] * 10)
    
    # Preview the results
    display(df[['Make', 'Model', 'Word_Count', 'Readability_Score', 'Composite_Effort_Score']].head())
else:
    print(f"Missing one of the required text columns: {text_cols_to_check}")

NameError: name 'SentimentIntensityAnalyzer' is not defined

## Export Results
Drops the massive raw text columns and saves the lightweight NLP features into the frontend data directory for quick loading in Streamlit.


In [ ]:
OUTPUT_BRANDS = "../data/frontend_data/nlp_brands.csv"
OUTPUT_ARCHETYPES = "../data/frontend_data/nlp_archetypes.csv"
OUTPUT_PERSONAS = "../data/frontend_data/nlp_personas.csv"
OUTPUT_EFFORT = "../data/frontend_data/nlp_effort_scores.csv"

# 1. Export Brands
brands_cols = ['URL', 'Make', 'Model', 'Sold_Price', 'Has_Premium_Mods', 'Premium_Brand_Count', 'Extracted_Brands_List']
if set(brands_cols).issubset(df.columns):
    df[brands_cols].to_csv(OUTPUT_BRANDS, index=False)
    print(f"Saved Brands to {OUTPUT_BRANDS}")

# 2. Export Archetypes
archetypes_cols = ['URL', 'Make', 'Model', 'Sold_Price', 'Archetype_Cluster']
if set(archetypes_cols).issubset(df.columns):
    df[archetypes_cols].to_csv(OUTPUT_ARCHETYPES, index=False)
    print(f"Saved Archetypes to {OUTPUT_ARCHETYPES}")

# 3. Export Personas (Only if you ran the full dataset in Cell 4)
personas_cols = ['URL', 'Make', 'Model', 'Sold_Price', 'Seller_Persona']
if set(personas_cols).issubset(df.columns):
    df[personas_cols].to_csv(OUTPUT_PERSONAS, index=False)
    print(f"Saved Personas to {OUTPUT_PERSONAS}")

# 4. Export Effort Scores
effort_cols = [
    'URL', 'Make', 'Model', 'Sold_Price', 
    'Word_Count', 'Readability_Score', 'Bullet_Point_Count', 
    'Sentiment_Score', 'Composite_Effort_Score'
]

if set(effort_cols).issubset(df.columns):
    df[effort_cols].to_csv(OUTPUT_EFFORT, index=False)
    print(f"Saved Effort Scores to {OUTPUT_EFFORT}")